**It takes around 6~7 hours to train a model with single fold.**

So, you should run this notebook four time (for each seed=42,43,44,45) to get all 4 seed weights of the model.

Or you can just press 'Save and Run All' to get the result of the fold 0 model.

There are two issues with this notebook:

- Training time issue: The weird thing is that it takes around 3 hours in Colab TPU (v2-8), which is supposed to be slower than Kaggle's TPU (v3-8). If you know a lot about tf+TPU frameworks, please let me know how to debug this issue.

- Training unstability: I Changed some minor configurations from the final solution. I changed epoch 400 -> 300, clipvalue=1. -> None, label_smoothing=0 -> label_smoothing=0.1 for more stable reproducibility(with slightly lower accuracy). unstability mainly caused by high lambda value of the AWP(0.2) - it will cause nan loss somtimes(around once out of five times). you can switch it to 0.1 or lower the learning rate and still can get fairly high accuracy. Also, if you have any idea related to this issue, please let me know.

In [1]:
!pip install -q /kaggle/input/tensorflow-2120/tensorflow-2.12.0-cp38-cp38-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
!pip install -q tensorflow-addons==0.20.0
!pip install -q git+https://github.com/hoyso48/tf-utils@main

ERROR: tensorflow-2.12.0-cp38-cp38-manylinux_2_17_x86_64.manylinux2014_x86_64.whl is not a supported wheel on this platform.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf 23.4.0 requires cupy-cuda11x<12.0.0a0,>=9.5.0, which is not installed.
onnx 1.13.1 requires protobuf<4,>=3.20.2, but you have protobuf 3.19.6 which is incompatible.
kfp 1.8.20 requires google-api-python-client<2,>=1.7.8, but you have google-api-python-client 2.86.0 which is incompatible.
kfp 1.8.20 requires PyYAML<6,>=5.3, but you have pyyaml 6.0 which is incompatible.
gcsfs 2023.3.0 requires fsspec==2023.3.0, but you have fsspec 2023.4.0 which is incompatible.
cudf 23.4.0 requires protobuf<4.22,>=4.21.6, but you have protobuf 3.19.6 which is incompatible.
beatrix-jupyterlab 2023.46.184821 requires jupyter-server~=1.16, but you have jupyter-server 2.5.0 which is incompatible.
apache-bea

In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_addons as tfa
import matplotlib.pyplot as plt
import matplotlib as mpl
import tensorflow.keras.mixed_precision as mixed_precision

from tqdm.autonotebook import tqdm
import sklearn

from tf_utils.schedules import OneCycleLR, ListedLR
from tf_utils.callbacks import Snapshot, SWA
from tf_utils.learners import FGM, AWP

import os
import time
import pickle
import math
import random
import sys
import cv2
import gc
import glob
import datetime

print(f'Tensorflow Version: {tf.__version__}')
print(f'Python Version: {sys.version}')

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/opt/conda/lib/python3.10/site-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
/tmp/ipykernel_37/1000973416.py:9: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


Tensorflow Version: 2.11.0
Python Version: 3.10.10 | packaged by conda-forge | (main, Mar 24 2023, 20:08:06) [GCC 11.3.0]


In [3]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

In [4]:
def get_strategy(device=None):

    IS_TPU = False

    try:
        # Try TPU first (only if explicitly requested)
        if device == "TPU":
            print("Connecting to TPU...")
            resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
            tf.config.experimental_connect_to_cluster(resolver)
            tf.tpu.experimental.initialize_tpu_system(resolver)
            strategy = tf.distribute.TPUStrategy(resolver)
            IS_TPU = True
            print("TPU connected.")

        else:
            raise ValueError("Skip TPU")

    except:
        # Fallback to GPU / CPU
        gpus = tf.config.list_physical_devices('GPU')

        if len(gpus) > 1:
            print("Using Multi-GPU")
            strategy = tf.distribute.MirroredStrategy()

        elif len(gpus) == 1:
            print("Using Single GPU")
            strategy = tf.distribute.get_strategy()

        else:
            print("Using CPU")
            strategy = tf.distribute.get_strategy()

    REPLICAS = strategy.num_replicas_in_sync
    print(f"REPLICAS: {REPLICAS}")

    return strategy, REPLICAS, IS_TPU
def get_strategy(device=None):

    IS_TPU = False

    try:
        # Try TPU first (only if explicitly requested)
        if device == "TPU":
            print("Connecting to TPU...")
            resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
            tf.config.experimental_connect_to_cluster(resolver)
            tf.tpu.experimental.initialize_tpu_system(resolver)
            strategy = tf.distribute.TPUStrategy(resolver)
            IS_TPU = True
            print("TPU connected.")

        else:
            raise ValueError("Skip TPU")

    except:
        # Fallback to GPU / CPU
        gpus = tf.config.list_physical_devices('GPU')

        if len(gpus) > 1:
            print("Using Multi-GPU")
            strategy = tf.distribute.MirroredStrategy()

        elif len(gpus) == 1:
            print("Using Single GPU")
            strategy = tf.distribute.get_strategy()

        else:
            print("Using CPU")
            strategy = tf.distribute.get_strategy()

    REPLICAS = strategy.num_replicas_in_sync
    print(f"REPLICAS: {REPLICAS}")

    return strategy, REPLICAS, IS_TPU


STRATEGY, N_REPLICAS, IS_TPU = get_strategy()

Using Single GPU
REPLICAS: 1


In [5]:
TRAIN_FILENAMES = glob.glob('/kaggle/input/islr-5fold/*.tfrecords')
print(len(TRAIN_FILENAMES))

187


In [6]:
# Train DataFrame
train_df = pd.read_csv('/kaggle/input/asl-signs/train.csv')
display(train_df.head())
display(train_df.info())

,path,participant_id,sequence_id,sign
0,train_landmark_files/26734/1000035562.parquet,26734,1000035562,blow
1,train_landmark_files/28656/1000106739.parquet,28656,1000106739,wait
2,train_landmark_files/16069/100015657.parquet,16069,100015657,cloud
3,train_landmark_files/25571/1000210073.parquet,25571,1000210073,bird
4,train_landmark_files/62590/1000240708.parquet,62590,1000240708,owie


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 94477 entries, 0 to 94476
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   path            94477 non-null  object
 1   participant_id  94477 non-null  int64 
 2   sequence_id     94477 non-null  int64 
 3   sign            94477 non-null  object
dtypes: int64(2), object(2)
memory usage: 2.9+ MB


None

In [7]:
import re
def count_data_items(filenames):
    n = [int(re.compile(r"-([0-9]*)\.").search(filename.split('/')[-1]).group(1)) for filename in filenames]
    return np.sum(n)
print(count_data_items(TRAIN_FILENAMES), len(train_df))
assert count_data_items(TRAIN_FILENAMES) == len(train_df)

94477 94477


In [8]:
ROWS_PER_FRAME = 543
MAX_LEN = 384
CROP_LEN = MAX_LEN
NUM_CLASSES  = 250
PAD = -100.
NOSE=[
    1,2,98,327
]
LNOSE = [98]
RNOSE = [327]
LIP = [ 0, 
    61, 185, 40, 39, 37, 267, 269, 270, 409,
    291, 146, 91, 181, 84, 17, 314, 405, 321, 375,
    78, 191, 80, 81, 82, 13, 312, 311, 310, 415,
    95, 88, 178, 87, 14, 317, 402, 318, 324, 308,
]
LLIP = [84,181,91,146,61,185,40,39,37,87,178,88,95,78,191,80,81,82]
RLIP = [314,405,321,375,291,409,270,269,267,317,402,318,324,308,415,310,311,312]

POSE = [500, 502, 504, 501, 503, 505, 512, 513]
LPOSE = [513,505,503,501]
RPOSE = [512,504,502,500]

REYE = [
    33, 7, 163, 144, 145, 153, 154, 155, 133,
    246, 161, 160, 159, 158, 157, 173,
]
LEYE = [
    263, 249, 390, 373, 374, 380, 381, 382, 362,
    466, 388, 387, 386, 385, 384, 398,
]

LHAND = np.arange(468, 489).tolist()
RHAND = np.arange(522, 543).tolist()

POINT_LANDMARKS = LIP + LHAND + RHAND + NOSE + REYE + LEYE #+POSE

NUM_NODES = len(POINT_LANDMARKS)
CHANNELS = 6*NUM_NODES

print(NUM_NODES)
print(CHANNELS)

def interp1d_(x, target_len, method='random'):
    length = tf.shape(x)[1]
    target_len = tf.maximum(1,target_len)
    if method == 'random':
        if tf.random.uniform(()) < 0.33:
            x = tf.image.resize(x, (target_len,tf.shape(x)[1]),'bilinear')
        else:
            if tf.random.uniform(()) < 0.5:
                x = tf.image.resize(x, (target_len,tf.shape(x)[1]),'bicubic')
            else:
                x = tf.image.resize(x, (target_len,tf.shape(x)[1]),'nearest')
    else:
        x = tf.image.resize(x, (target_len,tf.shape(x)[1]),method)
    return x

def tf_nan_mean(x, axis=0, keepdims=False):
    return tf.reduce_sum(tf.where(tf.math.is_nan(x), tf.zeros_like(x), x), axis=axis, keepdims=keepdims) / tf.reduce_sum(tf.where(tf.math.is_nan(x), tf.zeros_like(x), tf.ones_like(x)), axis=axis, keepdims=keepdims)

def tf_nan_std(x, center=None, axis=0, keepdims=False):
    if center is None:
        center = tf_nan_mean(x, axis=axis,  keepdims=True)
    d = x - center
    return tf.math.sqrt(tf_nan_mean(d * d, axis=axis, keepdims=keepdims))

class Preprocess(tf.keras.layers.Layer):
    def __init__(self, max_len=MAX_LEN, point_landmarks=POINT_LANDMARKS, **kwargs):
        super().__init__(**kwargs)
        self.max_len = max_len
        self.point_landmarks = point_landmarks

    def call(self, inputs):
        if tf.rank(inputs) == 3:
            x = inputs[None,...]
        else:
            x = inputs
        
        mean = tf_nan_mean(tf.gather(x, [17], axis=2), axis=[1,2], keepdims=True)
        mean = tf.where(tf.math.is_nan(mean), tf.constant(0.5,x.dtype), mean)
        x = tf.gather(x, self.point_landmarks, axis=2) #N,T,P,C
        std = tf_nan_std(x, center=mean, axis=[1,2], keepdims=True)
        
        x = (x - mean)/std

        if self.max_len is not None:
            x = x[:,:self.max_len]
        length = tf.shape(x)[1]
        x = x[...,:2]

        dx = tf.cond(tf.shape(x)[1]>1,lambda:tf.pad(x[:,1:] - x[:,:-1], [[0,0],[0,1],[0,0],[0,0]]),lambda:tf.zeros_like(x))

        dx2 = tf.cond(tf.shape(x)[1]>2,lambda:tf.pad(x[:,2:] - x[:,:-2], [[0,0],[0,2],[0,0],[0,0]]),lambda:tf.zeros_like(x))

        x = tf.concat([
            tf.reshape(x, (-1,length,2*len(self.point_landmarks))),
            tf.reshape(dx, (-1,length,2*len(self.point_landmarks))),
            tf.reshape(dx2, (-1,length,2*len(self.point_landmarks))),
        ], axis = -1)
        
        x = tf.where(tf.math.is_nan(x),tf.constant(0.,x.dtype),x)
        
        return x

118
708


In [ ]:
def decode_tfrec(record_bytes):
    features = tf.io.parse_single_example(record_bytes, {
        'coordinates': tf.io.FixedLenFeature([], tf.string),
        'sign': tf.io.FixedLenFeature([], tf.int64),
    })
    out = {}
    out['coordinates']  = tf.reshape(tf.io.decode_raw(features['coordinates'], tf.float32), (-1,ROWS_PER_FRAME,3))
    out['sign'] = features['sign']
    return out

def filter_nans_tf(x, ref_point=POINT_LANDMARKS):
    mask = tf.math.logical_not(tf.reduce_all(tf.math.is_nan(tf.gather(x,ref_point,axis=1)), axis=[-2,-1]))
    x = tf.boolean_mask(x, mask, axis=0)
    return x

def preprocess(x, augment=False, max_len=MAX_LEN):
    coord = x['coordinates']
    coord = filter_nans_tf(coord)
    if augment:
        coord = augment_fn(coord, max_len=max_len)
    coord = tf.ensure_shape(coord, (None,ROWS_PER_FRAME,3))
    
    return tf.cast(Preprocess(max_len=max_len)(coord)[0],tf.float32), tf.one_hot(x['sign'], NUM_CLASSES)

def flip_lr(x):
    x,y,z = tf.unstack(x, axis=-1)
    x = 1-x
    new_x = tf.stack([x,y,z], -1)
    new_x = tf.transpose(new_x, [1,0,2])
    lhand = tf.gather(new_x, LHAND, axis=0)
    rhand = tf.gather(new_x, RHAND, axis=0)
    new_x = tf.tensor_scatter_nd_update(new_x, tf.constant(LHAND)[...,None], rhand)
    new_x = tf.tensor_scatter_nd_update(new_x, tf.constant(RHAND)[...,None], lhand)
    llip = tf.gather(new_x, LLIP, axis=0)
    rlip = tf.gather(new_x, RLIP, axis=0)
    new_x = tf.tensor_scatter_nd_update(new_x, tf.constant(LLIP)[...,None], rlip)
    new_x = tf.tensor_scatter_nd_update(new_x, tf.constant(RLIP)[...,None], llip)
    lpose = tf.gather(new_x, LPOSE, axis=0)
    rpose = tf.gather(new_x, RPOSE, axis=0)
    new_x = tf.tensor_scatter_nd_update(new_x, tf.constant(LPOSE)[...,None], rpose)
    new_x = tf.tensor_scatter_nd_update(new_x, tf.constant(RPOSE)[...,None], lpose)
    leye = tf.gather(new_x, LEYE, axis=0)
    reye = tf.gather(new_x, REYE, axis=0)
    new_x = tf.tensor_scatter_nd_update(new_x, tf.constant(LEYE)[...,None], reye)
    new_x = tf.tensor_scatter_nd_update(new_x, tf.constant(REYE)[...,None], leye)
    lnose = tf.gather(new_x, LNOSE, axis=0)
    rnose = tf.gather(new_x, RNOSE, axis=0)
    new_x = tf.tensor_scatter_nd_update(new_x, tf.constant(LNOSE)[...,None], rnose)
    new_x = tf.tensor_scatter_nd_update(new_x, tf.constant(RNOSE)[...,None], lnose)
    new_x = tf.transpose(new_x, [1,0,2])
    return new_x

def resample(x, rate=(0.8,1.2)):
    rate = tf.random.uniform((), rate[0], rate[1])
    length = tf.shape(x)[0]
    new_size = tf.cast(rate*tf.cast(length,tf.float32), tf.int32)
    new_x = interp1d_(x, new_size)
    return new_x

def spatial_random_affine(xyz,
    scale  = (0.8,1.2),
    shear = (-0.15,0.15),
    shift  = (-0.1,0.1),
    degree = (-30,30),
):
    center = tf.constant([0.5,0.5])
    if scale is not None:
        scale = tf.random.uniform((),*scale)
        xyz = scale*xyz

    if shear is not None:
        xy = xyz[...,:2]
        z = xyz[...,2:]
        shear_x = shear_y = tf.random.uniform((),*shear)
        if tf.random.uniform(()) < 0.5:
            shear_x = 0.
        else:
            shear_y = 0.
        shear_mat = tf.identity([
            [1.,shear_x],
            [shear_y,1.]
        ])
        xy = xy @ shear_mat
        center = center + [shear_y, shear_x]
        xyz = tf.concat([xy,z], axis=-1)

    if degree is not None:
        xy = xyz[...,:2]
        z = xyz[...,2:]
        xy -= center
        degree = tf.random.uniform((),*degree)
        radian = degree/180*np.pi
        c = tf.math.cos(radian)
        s = tf.math.sin(radian)
        rotate_mat = tf.identity([
            [c,s],
            [-s, c],
        ])
        xy = xy @ rotate_mat
        xy = xy + center
        xyz = tf.concat([xy,z], axis=-1)

    if shift is not None:
        shift = tf.random.uniform((),*shift)
        xyz = xyz + shift

    return xyz

def temporal_crop(x, length=MAX_LEN):
    l = tf.shape(x)[0]
    offset = tf.random.uniform((), 0, tf.clip_by_value(l-length,1,length), dtype=tf.int32)
    x = x[offset:offset+length]
    return x

def temporal_mask(x, size=(0.2,0.4), mask_value=float('nan')):
    l = tf.shape(x)[0]
    mask_size = tf.random.uniform((), *size)
    mask_size = tf.cast(tf.cast(l, tf.float32) * mask_size, tf.int32)
    mask_offset = tf.random.uniform((), 0, tf.clip_by_value(l-mask_size,1,l), dtype=tf.int32)
    x = tf.tensor_scatter_nd_update(x,tf.range(mask_offset, mask_offset+mask_size)[...,None],tf.fill([mask_size,543,3],mask_value))
    return x

def spatial_mask(x, size=(0.2,0.4), mask_value=float('nan')):
    mask_offset_y = tf.random.uniform(())
    mask_offset_x = tf.random.uniform(())
    mask_size = tf.random.uniform((), *size)
    mask_x = (mask_offset_x<x[...,0]) & (x[...,0] < mask_offset_x + mask_size)
    mask_y = (mask_offset_y<x[...,1]) & (x[...,1] < mask_offset_y + mask_size)
    mask = mask_x & mask_y
    x = tf.where(mask[...,None], mask_value, x)
    return x

def augment_fn(x, always=False, max_len=None):
    if tf.random.uniform(())<0.8 or always:
        x = resample(x, (0.5,1.5))
    if tf.random.uniform(())<0.5 or always:
        x = flip_lr(x)
    if max_len is not None:
        x = temporal_crop(x, max_len)
    if tf.random.uniform(())<0.75 or always:
        x = spatial_random_affine(x)
    if tf.random.uniform(())<0.5 or always:
        x = temporal_mask(x)
    if tf.random.uniform(())<0.5 or always:
        x = spatial_mask(x)
    return x

def get_tfrec_dataset(tfrecords, batch_size=64, max_len=64, drop_remainder=False, augment=False, shuffle=False, repeat=False):
    # Initialize dataset with TFRecords
    ds = tf.data.TFRecordDataset(tfrecords, num_parallel_reads=tf.data.AUTOTUNE, compression_type='GZIP')
    ds = ds.map(decode_tfrec, tf.data.AUTOTUNE)
    ds = ds.map(lambda x: preprocess(x, augment=augment, max_len=max_len), tf.data.AUTOTUNE)

    if repeat: 
        ds = ds.repeat()
        
    if shuffle:
        ds = ds.shuffle(shuffle)
        options = tf.data.Options()
        options.experimental_deterministic = (False)
        ds = ds.with_options(options)
    
    if batch_size:
        ds = ds.padded_batch(batch_size, padding_values=PAD, padded_shapes=([max_len,CHANNELS],[NUM_CLASSES]), drop_remainder=drop_remainder)

    ds = ds.prefetch(tf.data.AUTOTUNE)
        
    return ds

ds = get_tfrec_dataset(TRAIN_FILENAMES, augment=True, batch_size=1024)
for x in ds:
    temp_train = x
    break

In [ ]:
from IPython.display import HTML
import matplotlib.animation as animation
from matplotlib.animation import FuncAnimation

def filter_nans(frames):
    return frames[~np.isnan(frames).all(axis=(-2,-1))]

ds = tf.data.TFRecordDataset(TRAIN_FILENAMES, num_parallel_reads=tf.data.AUTOTUNE, compression_type='GZIP')
ds = ds.map(decode_tfrec, tf.data.AUTOTUNE)
print(ds)
for x in ds:
    temp = x['coordinates'].numpy()
    if not len(filter_nans(temp[:,LHAND])) == 0:
        break
    
edges = [(0,1),(1,2),(2,3),(3,4),(0,5),(0,17),(5,6),(6,7),(7,8),(5,9),(9,10),(10,11),(11,12),
         (9,13),(13,14),(14,15),(15,16),(13,17),(17,18),(18,19),(19,20)]

fig, ax = plt.subplots()

def plot_frame(frame, edges=[], idxs=[]):
        
    frame[np.isnan(frame)] = 0
    x = list(frame[...,0])
    y = list(frame[...,1])
    if len(idxs) == 0:
        idxs = list(range(len(x)))
    ax.clear()
    ax.scatter(x, y, color='dodgerblue')
    for i in range(len(x)):
        ax.text(x[i], y[i], idxs[i])
        
    for edge in edges:
        ax.plot([x[edge[0]], x[edge[1]]], [y[edge[0]], y[edge[1]]], color='salmon')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xticklabels([])
    ax.set_yticklabels([])

def animate_frames(frames, edges=[], idxs=[]):
    anim = FuncAnimation(fig, lambda frame: plot_frame(frame, edges, idxs), frames=frames, interval=100)
    return HTML(anim.to_jshtml())

In [ ]:
# ===============================
# Load Dataset and Inspect X & y
# ===============================

# Create dataset without augmentation for clean inspection
ds = get_tfrec_dataset(
    TRAIN_FILENAMES,
    batch_size=32,          # small batch for inspection
    max_len=MAX_LEN,
    augment=False,
    shuffle=False
)

# Take one batch from dataset
for batch in ds.take(1):
    X, y = batch  # Separate features and labels

# ===============================
# Print Basic Information
# ===============================

print("X shape:", X.shape)  # (batch_size, time_steps, features)
print("y shape:", y.shape)  # (batch_size, num_classes)

print("\nData type of X:", X.dtype)
print("Data type of y:", y.dtype)

# ===============================
# Inspect One Sample
# ===============================

sample_X = X[0]
sample_y = y[0]

print("\nSingle sample shape:", sample_X.shape)
print("Single label shape:", sample_y.shape)

print("\nNumber of time steps:", sample_X.shape[0])
print("Number of features per frame:", sample_X.shape[1])

print("\nClass index of first sample:", tf.argmax(sample_y).numpy())

# ===============================
# Check Value Statistics
# ===============================

print("\nX min value:", tf.reduce_min(X).numpy())
print("X max value:", tf.reduce_max(X).numpy())
print("X mean value:", tf.reduce_mean(X).numpy())

In [ ]:
# ============================================
# Train / Validation Split using TFRecord files
# ============================================

from sklearn.model_selection import train_test_split

# Split TFRecord files (file-level split)
train_files, valid_files = train_test_split(
    TRAIN_FILENAMES,
    test_size=0.2,       # 20% validation
    random_state=42,
    shuffle=True
)

print("Number of training TFRecords:", len(train_files))
print("Number of validation TFRecords:", len(valid_files))

# ============================================
# Build Training Dataset
# ============================================

train_ds = get_tfrec_dataset(
    train_files,
    batch_size=128,
    max_len=MAX_LEN,
    augment=True,        # Enable augmentation for training
    shuffle=2048,
    repeat=True
)

# ============================================
# Build Validation Dataset
# ============================================

valid_ds = get_tfrec_dataset(
    valid_files,
    batch_size=128,
    max_len=MAX_LEN,
    augment=False,       # No augmentation for validation
    shuffle=False,
    repeat=False
)

# ============================================
# Inspect One Batch
# ============================================

for X_batch, y_batch in train_ds.take(1):
    print("\nTrain batch shape:", X_batch.shape)
    print("Train label shape:", y_batch.shape)
    break

for X_val, y_val in valid_ds.take(1):
    print("\nValidation batch shape:", X_val.shape)
    print("Validation label shape:", y_val.shape)
    break# ============================================
# Train / Validation Split using TFRecord files
# ============================================

from sklearn.model_selection import train_test_split

# Split TFRecord files (file-level split)
train_files, valid_files = train_test_split(
    TRAIN_FILENAMES,
    test_size=0.2,       # 20% validation
    random_state=42,
    shuffle=True
)

print("Number of training TFRecords:", len(train_files))
print("Number of validation TFRecords:", len(valid_files))

# ============================================
# Build Training Dataset
# ============================================

train_ds = get_tfrec_dataset(
    train_files,
    batch_size=128,
    max_len=MAX_LEN,
    augment=True,        # Enable augmentation for training
    shuffle=2048,
    repeat=True
)

# ============================================
# Build Validation Dataset
# ============================================

valid_ds = get_tfrec_dataset(
    valid_files,
    batch_size=128,
    max_len=MAX_LEN,
    augment=False,       # No augmentation for validation
    shuffle=False,
    repeat=False
)

# ============================================
# Inspect One Batch
# ============================================

for X_batch, y_batch in train_ds.take(1):
    print("\nTrain batch shape:", X_batch.shape)
    print("Train label shape:", y_batch.shape)
    break

for X_val, y_val in valid_ds.take(1):
    print("\nValidation batch shape:", X_val.shape)
    print("Validation label shape:", y_val.shape)
    break

In [ ]:
# ==========================================
# Save Correct Label Mapping (100% Safe)
# ==========================================

import json
import pandas as pd
import tensorflow as tf

# Load CSV (already loaded usually)
train_df = pd.read_csv('/kaggle/input/asl-signs/train.csv')

# Use the SAME ordering assumption used in TFRecord creation
class_names = sorted(train_df['sign'].unique())

# Build id -> label mapping
id_to_label = {i: label for i, label in enumerate(class_names)}

# Optional safety check (confirm label range inside TFRecords)
raw_ds = tf.data.TFRecordDataset(TRAIN_FILENAMES, compression_type='GZIP')
raw_ds = raw_ds.map(decode_tfrec)

label_ids = set()
for sample in raw_ds.take(2000):   # check first 2000 samples فقط
    label_ids.add(int(sample['sign'].numpy()))

assert min(label_ids) == 0
assert max(label_ids) == len(class_names) - 1

# Save JSON
with open("labels.json", "w") as f:
    json.dump(id_to_label, f, indent=4)

print("labels.json saved successfully and verified.")

In [ ]:
# =====================================================
# Pure Strong BiLSTM Model (No Attention)
# =====================================================

def build_model(input_dim=CHANNELS, num_classes=NUM_CLASSES):

    inputs = tf.keras.Input(shape=(None, input_dim))

    # Ignore padding
    x = tf.keras.layers.Masking(mask_value=PAD)(inputs)

    # -------------------------
    # BiLSTM Block 1
    # -------------------------
    x1 = tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(
            256,
            return_sequences=True,
            dropout=0.3
        )
    )(x)

    x1 = tf.keras.layers.LayerNormalization()(x1)

    # -------------------------
    # BiLSTM Block 2
    # -------------------------
    x2 = tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(
            256,
            return_sequences=True,
            dropout=0.3
        )
    )(x1)

    x2 = tf.keras.layers.LayerNormalization()(x2)

    # Residual connection
    x = tf.keras.layers.Add()([x1, x2])

    # -------------------------
    # Final LSTM (sequence → vector)
    # -------------------------
    x = tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(
            128,
            return_sequences=False,
            dropout=0.3
        )
    )(x)

    # -------------------------
    # Classification Head
    # -------------------------
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.4)(x)

    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs)

    return model

In [ ]:
import tensorflow_addons as tfa

In [ ]:
with STRATEGY.scope():

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=3e-4,
        clipnorm=1.0
    )

    model = build_model()

    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=["accuracy"]
    )

model.summary()

In [ ]:
# ==========================================
# Callbacks (Stable for long training)
# ==========================================

callbacks_list = [

    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=20,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        'best_asl_model.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

In [ ]:
EPOCHS = 30
BATCH_SIZE = 64 * N_REPLICAS

TOTAL_TRAIN_SAMPLES = count_data_items(train_files)
TOTAL_VAL_SAMPLES = count_data_items(valid_files)

steps_per_epoch = TOTAL_TRAIN_SAMPLES // BATCH_SIZE
validation_steps = TOTAL_VAL_SAMPLES // BATCH_SIZE

print("Steps per epoch:", steps_per_epoch)
print("Validation steps:", validation_steps)

In [ ]:
# ==========================================
# Start Training
# ==========================================

history = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=callbacks_list,
    verbose=1
)

In [ ]:
print("\nEvaluating on Validation Set...")
loss, acc = model.evaluate(valid_ds, steps=validation_steps)
print(f"Validation Accuracy: {acc*100:.2f}%")

In [ ]:
# ==========================================
# Training Summary
# ==========================================

print("Best Training Accuracy:", max(history.history['accuracy']))
print("Best Validation Accuracy:", max(history.history['val_accuracy']))

best_epoch = history.history['val_accuracy'].index(max(history.history['val_accuracy'])) + 1
print("Best Epoch:", best_epoch)

print("\nFinal Training Loss:", history.history['loss'][-1])
print("Final Validation Loss:", history.history['val_loss'][-1])

In [ ]:
# ==========================================
# Training Curves
# ==========================================

import matplotlib.pyplot as plt

plt.figure(figsize=(14,5))

# Accuracy
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

In [ ]:
# ==========================================
# Model Evaluation
# ==========================================

val_loss, val_acc = model.evaluate(valid_ds, steps=validation_steps)

print(f"Validation Accuracy: {val_acc*100:.2f}%")
print(f"Validation Loss: {val_loss:.4f}")

In [ ]:
# ==========================================
# Get Predictions
# ==========================================

import numpy as np

y_true = []
y_pred = []

for X_batch, y_batch in valid_ds.take(validation_steps):
    
    preds = model.predict(X_batch, verbose=0)
    
    y_true.extend(np.argmax(y_batch.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("Total evaluated samples:", len(y_true))

In [ ]:
# ==========================================
# Confusion Matrix
# ==========================================

from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(12,10))
sns.heatmap(cm, cmap='Blues')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

In [ ]:
# ==========================================
# Classification Report
# ==========================================

from sklearn.metrics import classification_report

report = classification_report(y_true, y_pred)
print(report)

In [ ]:
# ==========================================
# Build Class Mapping (Correct Order)
# ==========================================

unique_signs = sorted(train_df['sign'].unique())

print("Number of classes:", len(unique_signs))

class_names = unique_signs

# Create label_map (name → id)
label_map = {label: idx for idx, label in enumerate(class_names)}

# Reverse map (id → name)
id_to_label = {v: k for k, v in label_map.items()}

In [ ]:
report = classification_report(y_true, y_pred, target_names=class_names)
print(report)

In [ ]:
from sklearn.metrics import classification_report
import json

report_dict = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    output_dict=True
)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

summary_metrics = {
    "overall_accuracy": float(accuracy_score(y_true, y_pred)),
    "macro_precision": float(precision_score(y_true, y_pred, average="macro")),
    "macro_recall": float(recall_score(y_true, y_pred, average="macro")),
    "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
    "total_parameters": int(model.count_params()),
    "num_classes": int(NUM_CLASSES)
}
full_report = {
    "summary": summary_metrics,
    "per_class_metrics": report_dict
}

In [ ]:
with open("asl_model_evaluation_report.json", "w") as f:
    json.dump(full_report, f, indent=4)

print("Evaluation report saved as asl_model_evaluation_report.json")

In [ ]:
# ==========================================
# Precision / Recall / F1 Visualization
# ==========================================

from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_true, y_pred, average='macro')
recall = recall_score(y_true, y_pred, average='macro')
f1 = f1_score(y_true, y_pred, average='macro')

metrics = ['Precision', 'Recall', 'F1-score']
values = [precision, recall, f1]

plt.figure(figsize=(6,4))
plt.bar(metrics, values)
plt.title("Macro Metrics")
plt.ylim(0,1)
plt.show()

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

In [ ]:
# ==========================================
# Most Confused Classes
# ==========================================

misclassified = y_true != y_pred

print("Total Misclassified Samples:", np.sum(misclassified))
print("Error Rate:", np.mean(misclassified))

In [ ]:
# ==========================================
# Model Complexity
# ==========================================

model.summary()

total_params = model.count_params()
print("Total Parameters:", total_params)

In [ ]:
# ==========================================
# Prediction Distribution
# ==========================================

plt.figure(figsize=(8,4))
plt.hist(y_pred, bins=50)
plt.title("Prediction Distribution")
plt.show()

In [ ]:
# Save full model in TensorFlow format
model.save("asl_model_saved")

In [ ]:
# Convert trained model to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Important for LSTM / Attention layers
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
converter._experimental_lower_tensor_list_ops = False

# Convert
tflite_model = converter.convert()

# Save file
with open("asl_model.tflite", "wb") as f:
    f.write(tflite_model)

print("Model saved as asl_model.tflite")